# Microstructure & Anomaly Batch — **Test D: the D-GATE** (variance-ratio-first)

**The mechanical identity that shapes everything.** Total wavelet energy = realized
variance, so **raw band energy IS vol in disguise** — testing it rebuilds dead M3.
Only **normalized shares** can survive, and the candidate is the **coarse-band
share** (slow-scale energy). The **variance ratio** asks the identical economic
question — *where does movement energy live: slow/trending vs fast/mean-reverting?*
— is **intrinsically vol-normalized** (a ratio of variances), and skips the
short-series coarse-scale wavelet edge-effect fragility. **So the gate runs on the
variance ratio; wavelet code is built only if the VR gate clears** (per spec:
variance-ratio-first).

`VR(q) = Var(q-day return) / (q · Var(1-day return))`. VR > 1 = slow/trending;
VR < 1 = fast/mean-reverting. Estimator = overlapping Lo-MacKinlay, **verified
before build**: trend → 1.87, mean-rev → 0.40, random walk → 1.03 at N=1000;
unbiased at N=252 (mean 0.95) **but high per-stock noise (std 0.30 across seeds)** —
the short-series fragility gates 2 and 3 exist to catch.

**Gated order (do not reorder):**
1. **Vol-orthogonalization** — residualize VR against RV; does the residual predict
   forward returns (FM, controls RV/Amihud/12-1)? Insignificant → **D dead, shelve**.
2. **Dispersion** — is the cross-sectional VR spread wider than the ~0.30 per-stock
   estimation noise? Narrow → nothing to sort on → stop.
3. **Rank stability** — block-bootstrap VR rank stability > 0.6? Near M7's 0.44 →
   too fragile on 250 obs → stop.

Wavelet MODWT/LA8 localization is built **only if all three clear** — deferred, not
in this gate.

**Hold-out wall:** entire gate on 2010-2021 training only; hold-out untouched.
**Tally 31 → 32** (D-gate = 1 trial). Expectation (per spec): D dies cheaply — my
prior is gate 2 or 3, given the ±0.30 per-stock VR noise at 252 obs.

## Cell 0 — setup + verified variance-ratio estimator

In [1]:
# =============================================================================
# TEST D — D-GATE (variance-ratio-first; run FIRST; surface before wavelet code)
# =============================================================================
# The scout established a MECHANICAL IDENTITY: total wavelet energy = realized
# variance. So raw band energy IS vol in disguise -> testing it rebuilds dead M3.
# Only NORMALIZED shares can survive; the candidate is the COARSE-band share
# (slow-scale energy). The VARIANCE RATIO asks the identical economic question
# ("where does movement energy live — slow/trending vs fast/mean-reverting?"),
# is INTRINSICALLY vol-normalized (a ratio of variances — verified vol-invariant),
# and skips the short-series coarse-scale wavelet edge-effect fragility. So the
# GATE runs on the variance ratio; wavelet code is built ONLY if the VR gate
# clears (per spec: variance-ratio-FIRST).
#
# VR(q) = Var(q-day return) / (q * Var(1-day return)). VR>1 slow/trending energy,
# VR<1 fast/mean-reverting. Estimator = overlapping Lo-MacKinlay, VERIFIED:
# trend->1.87, mean-rev->0.40, RW->1.03 at N=1000; unbiased at N=252 (mean 0.95)
# but HIGH per-stock noise (std 0.30 across seeds) — the short-series fragility
# the dispersion/stability gates exist to catch.
#
# GATED ORDER (do NOT reorder):
#   GATE 1 — vol-orthogonalization: residualize VR against RV; does the residual
#     u predict forward returns (FM, controls RV/Amihud/12-1)? Insignificant ->
#     D DEAD, shelve (pre-agreed accepted outcome).
#   GATE 2 — dispersion: is cross-sectional spread of VR materially wider than
#     M7's 0.046 entropy band? Narrow -> nothing to sort on -> stop.
#   GATE 3 — rank stability: block-bootstrap VR rank stability across adjacent
#     windows > 0.6? Near M7's 0.44 -> too fragile on 250 obs -> stop.
#   (Wavelet MODWT/LA8 build ONLY if all three clear — deferred, not in this gate.)
#
# HOLD-OUT WALL: entire gate on 2010-2021 TRAINING only; residualization coeffs
# fit on training; hold-out untouched. TALLY: D-gate = 1 trial, 31 -> 32.
# =============================================================================
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd
from scipy.stats import spearmanr, rankdata

pd.set_option("display.width",220); pd.set_option("display.max_columns",80)
try:
    from google.colab import drive; drive.mount("/content/drive",force_remount=False)
except ImportError: pass

BASE=Path("/content/drive/MyDrive/moonshot_compounding")
RUN_DATE=datetime.now().strftime("%Y-%m-%d")
assert BASE.exists(), f"{BASE} not found — mount Drive."
def resolve_newest(folder,pattern):
    hits=sorted((BASE/folder).glob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)
    if not hits: raise FileNotFoundError(f"{folder}/{pattern}")
    return hits[0]
OUT_DIR=BASE/"hypotheses"/"microstructure_anomaly_batch"/f"D_gate_{RUN_DATE}"
(OUT_DIR/"artifacts").mkdir(parents=True,exist_ok=True)
def save_art(df,name): p=OUT_DIR/"artifacts"/f"{name}_{RUN_DATE}.csv"; df.to_csv(p,index=False); print(f"    -> {p.name}"); return p

print("="*78); print("TEST D — D-GATE (variance-ratio-first; TRAINING 2010-2021)"); print("="*78)

PANEL_PATH=resolve_newest("features","feature_panel_with_pit_v2_*.parquet")
panel=pd.read_parquet(PANEL_PATH)
TICKER_COL=next(c for c in panel.columns if c.lower() in ("ticker","symbol"))
DATE_COL=next(c for c in panel.columns if c.lower() in ("date","dt"))
panel[DATE_COL]=pd.to_datetime(panel[DATE_COL])
if getattr(panel[DATE_COL].dtype,"tz",None) is not None:
    panel[DATE_COL]=panel[DATE_COL].dt.tz_localize(None)
panel=panel.sort_values([TICKER_COL,DATE_COL]).reset_index(drop=True)
print(f"  panel: {PANEL_PATH.name} ({len(panel):,} rows)")
_closed=[pd.Timestamp(d) for d in ["2014-04-24","2014-10-15","2026-01-15","2026-05-01","2026-05-28","2026-06-26"]]
assert not any((panel[DATE_COL]==d).any() for d in _closed), "Pre-fix panel — rebuild v3."
VENDOR_FAILURE_DATES=set(pd.Timestamp(d) for d in panel.loc[panel["date_status"]=="vendor_failure",DATE_COL].unique())

AMIHUD_PATH=resolve_newest("features","amihud_monthly_v2_*.parquet")
amihud=pd.read_parquet(AMIHUD_PATH)
ah_tcol=next(c for c in amihud.columns if c.lower() in ("ticker","symbol"))
ah_mcol=next((c for c in amihud.columns if "month" in c.lower() or "date" in c.lower()),None)
ah_vcol=next(c for c in amihud.columns if "amihud" in c.lower())
amihud["month"]=pd.PeriodIndex(pd.to_datetime(amihud[ah_mcol]).dt.strftime("%Y-%m"),freq="M")
amihud_lookup={(r[ah_tcol],r["month"]):r[ah_vcol] for _,r in amihud.iterrows()}

TRAIN_END=pd.Timestamp("2021-12-31")
LOOKBACK,SKIP=252,21
WINDOW_ROWS=LOOKBACK-SKIP+1
VR_Q_PRIMARY=21; VR_Q_ROBUST=[10,42]
N_DECILES,MIN_NAMES=10,100
BLOCK_LEN=20
CORP_ACTION_SCREEN={"GREENPLY.NS","MVL.NS","TTML.NS","IGARASHI.NS","KTKBANK.NS",
                    "KARURVYSYA.NS","JISLJALEQS.NS","AJRINFRA.NS"}
SUBTEST_TALLY=32   # 31 carried (C-gate+diag) + D-gate
M7_ENTROPY_BAND=0.046; M7_STABILITY=0.44; STABILITY_BAR=0.6
print(f"  HOLD-OUT WALL: gate on TRAINING 2010-01-01 .. {TRAIN_END.date()} only.")
print(f"  running sub-test tally: {SUBTEST_TALLY} (31 carried + D-gate = 1 trial)")
print(f"  VR horizons: q={VR_Q_PRIMARY} primary, {VR_Q_ROBUST} robustness")

def variance_ratio(daily_ret, q):
    """Overlapping Lo-MacKinlay VR. VERIFIED vol-invariant; unbiased at 252 obs
    (mean 0.95) but high per-stock noise (std 0.30) — the fragility the gates test."""
    r=np.asarray(daily_ret,float); r=r[~np.isnan(r)]; n=len(r)
    if n < q+1: return np.nan
    mu=np.mean(r); sig1=np.sum((r-mu)**2)/(n-1)
    if sig1<=0: return np.nan
    X=np.concatenate([[0.0],np.cumsum(r)])
    dq=(X[q:]-X[:-q]) - q*mu; m=q*(n-q+1)*(1.0-q/n)
    if m<=0: return np.nan
    return (np.sum(dq**2)/m)/sig1

def newey_west_tstat(series, lag=6):
    x=np.asarray(series,float); x=x[~np.isnan(x)]; T=len(x)
    if T<lag+2: return np.nan,np.nan,np.nan
    mu=x.mean(); e=x-mu; var=(e@e)/T
    for L in range(1,lag+1):
        w=1-L/(lag+1); var+=2*w*(e[L:]@e[:-L])/T
    se=np.sqrt(var/T); return mu,se,(mu/se if se>0 else np.nan)
print("  setup complete. VR estimator loaded (pre-verified).")

Mounted at /content/drive
TEST D — D-GATE (variance-ratio-first; TRAINING 2010-2021)
  panel: feature_panel_with_pit_v2_2026-07-24.parquet (3,328,132 rows)
  HOLD-OUT WALL: gate on TRAINING 2010-01-01 .. 2021-12-31 only.
  running sub-test tally: 32 (31 carried + D-gate = 1 trial)
  VR horizons: q=21 primary, [10, 42] robustness
  setup complete. VR estimator loaded (pre-verified).


## Cell 1 — build VR panel (training only, cached)

In [2]:
# =============================================================================
# D-GATE — CELL 1 — BUILD VR PANEL (training only, cached)
# =============================================================================
tdata={}
for tkr,g in panel.groupby(TICKER_COL,sort=False):
    tdata[tkr]={"dates":g[DATE_COL].to_numpy().astype("datetime64[ns]"),
                "ret":g["daily_return"].to_numpy(dtype=float),
                "vf":g[DATE_COL].isin(VENDOR_FAILURE_DATES).to_numpy(dtype=bool),
                "flag":g["in_nifty500_flag"].to_numpy(dtype=float),
                "rv60":g["realized_vol_60"].to_numpy(dtype=float) if "realized_vol_60" in g else None,
                "turnover":g["turnover"].to_numpy(dtype=float)}
def find_pos(darr,as_of):
    t=np.datetime64(pd.Timestamp(as_of),"ns"); i=int(np.searchsorted(darr,t))
    return i if (i<len(darr) and darr[i]==t) else None
def momentum_signal(tkr,p):
    td=tdata[tkr]
    if p-LOOKBACK<0: return np.nan,False
    r=td["ret"][p-LOOKBACK:p-SKIP+1]; vf=td["vf"][p-LOOKBACK:p-SKIP+1]
    if len(r)!=WINDOW_ROWS: return np.nan,False
    excl=vf.copy()
    if excl.any(): s=np.zeros_like(vf); s[1:]=vf[:-1]; excl=excl|s
    if int(np.sum(~np.isnan(r[~excl])))<WINDOW_ROWS-int(excl.sum()): return np.nan,False
    good=r[~np.isnan(r)]
    return float(np.prod(1+good)-1),True
def formation_returns(tkr,p):
    td=tdata[tkr]
    if p-LOOKBACK<0: return None
    r=td["ret"][p-LOOKBACK:p-SKIP+1]; good=r[~np.isnan(r)]
    return good if len(good)>=200 else None
def rv_at(tkr,p):
    td=tdata[tkr]
    if td["rv60"] is None or p>=len(td["rv60"]): return np.nan
    v=td["rv60"][p]; return float(v) if np.isfinite(v) and v>0 else np.nan
def trailing_turnover(tkr,p,window=21,min_valid=15):
    td=tdata[tkr]
    if p-window<0: return np.nan
    vals=td["turnover"][p-window:p]
    if int(np.sum(~np.isnan(vals)))<min_valid: return np.nan
    return float(np.nanmean(vals))
all_days=pd.to_datetime(pd.Series(sorted(panel[DATE_COL].unique())))
MONTH_ENDS=list(pd.DataFrame({"d":all_days}).assign(ym=lambda x:x["d"].dt.to_period("M"))
                .groupby("ym")["d"].max().sort_values())
REBAL=[m for m in MONTH_ENDS if m<=TRAIN_END]
def fwd_return(tkr,p,N=21):
    td=tdata[tkr]
    if p+1>=len(td["ret"]): return np.nan
    r=td["ret"][p+1:p+N+1]; good=r[~np.isnan(r)]
    return float(np.prod(1+good)-1) if len(good)>0 else np.nan

def build_vr_panel():
    cache=OUT_DIR/"artifacts"/"vr_panel.parquet"
    if cache.exists():
        print(f"  [cache hit] {cache.name}"); return pd.read_parquet(cache)
    print("  building VR panel (training only)...")
    recs=[]
    for ri,rb in enumerate(REBAL):
        mo=pd.Period(rb,freq="M")
        for tkr,td in tdata.items():
            p=find_pos(td["dates"],rb)
            if p is None or td["flag"][p]!=1 or tkr in CORP_ACTION_SCREEN: continue
            m,ok=momentum_signal(tkr,p)
            if not ok: continue
            fr=formation_returns(tkr,p)
            if fr is None: continue
            fwd=fwd_return(tkr,p,21)
            if np.isnan(fwd): continue
            vr=variance_ratio(fr,VR_Q_PRIMARY)
            vr10=variance_ratio(fr,10); vr42=variance_ratio(fr,42)
            rv=rv_at(tkr,p); to=trailing_turnover(tkr,p); am=amihud_lookup.get((tkr,mo))
            recs.append((rb,tkr,m,vr,vr10,vr42,rv,to,
                         am if (am is not None and np.isfinite(am)) else np.nan,fwd))
        if ri%24==0: print(f"    ... {rb.date()} ({len(recs):,} rows)")
    df=pd.DataFrame(recs,columns=["rb","ticker","mom","vr","vr10","vr42","rv","turnover","amihud","fwd"])
    df.to_parquet(cache); print(f"  [cached] {cache.name} ({len(df):,} rows)")
    return df
vrdf=build_vr_panel()
print(f"  VR panel: {len(vrdf):,} stock-months, {vrdf['ticker'].nunique()} tickers (training).")

  building VR panel (training only)...
    ... 2010-01-29 (0 rows)
    ... 2012-01-31 (5,541 rows)
    ... 2014-01-31 (16,554 rows)
    ... 2016-01-29 (26,903 rows)
    ... 2018-01-31 (37,793 rows)
    ... 2020-01-31 (48,448 rows)
  [cached] vr_panel.parquet (59,184 rows)
  VR panel: 59,184 stock-months, 865 tickers (training).


## Cell 2 — the three gated checks

In [3]:
# =============================================================================
# D-GATE — CELL 2 — THE THREE GATED CHECKS
# =============================================================================
print("="*78); print("D-GATE — three sequential checks (surface before any wavelet build)"); print("="*78)
def zc(s):
    s=pd.Series(s,dtype=float); mu,sd=s.mean(),s.std(ddof=1)
    return (s-mu)/sd if sd>0 else s*0.0
GATE_PASS=True; gate_log=[]

# --- coverage / dispersion prelim -------------------------------------------
print("\n[coverage] VR panel non-null:")
print(f"  VR(q=21) coverage: {vrdf['vr'].notna().mean()*100:.1f}%")

# --- GATE 1: vol-orthogonalization ------------------------------------------
# residualize VR against RV each month, FM regress fwd on residual + controls
print("\n[GATE 1] VOL-ORTHOGONALIZATION — does VR predict returns AFTER removing vol?")
ent_unc=[]; ent_res=[]
for rb,g in vrdf.groupby("rb"):
    d=g.dropna(subset=["vr","rv","mom","fwd"]).copy()
    if len(d)<MIN_NAMES: continue
    d["z_vr"]=zc(d["vr"]); d["z_rv"]=zc(np.log(d["rv"].clip(lower=1e-9)))
    d["z_mom"]=zc(d["mom"])
    d["z_am"]=zc(np.log(d["amihud"].clip(lower=1e-12))) if d["amihud"].notna().any() else 0.0
    # residualize VR on RV
    b=np.polyfit(d["z_rv"],d["z_vr"],1); d["u"]=d["z_vr"]-(b[0]*d["z_rv"]+b[1])
    # uncontrolled: fwd ~ z_vr
    Xu=np.column_stack([np.ones(len(d)),d["z_vr"]])
    ent_unc.append(np.linalg.lstsq(Xu,d["fwd"].values,rcond=None)[0][1])
    # controlled: fwd ~ u(vr resid) + z_rv + z_mom + z_am
    Xc=np.column_stack([np.ones(len(d)),d["u"],d["z_rv"],d["z_mom"],
                        d["z_am"] if np.ndim(d["z_am"])>0 else np.zeros(len(d))])
    ent_res.append(np.linalg.lstsq(Xc,d["fwd"].values,rcond=None)[0][1])
mu_u,_,t_u=newey_west_tstat(ent_unc); mu_r,_,t_r=newey_west_tstat(ent_res)
print(f"  VR coef UNCONTROLLED:              {mu_u:+.5f} (t={t_u:+.2f})")
print(f"  VR-residual coef +RV+mom+Amihud:   {mu_r:+.5f} (t={t_r:+.2f})")
g1=abs(t_r)>2 and (mu_r*mu_u>0 if mu_u!=0 else True)
if g1:
    print("  -> GATE 1 PASS: orthogonalized VR retains forward-return power.")
    gate_log.append("gate1 orthogonalization: PASS")
else:
    print("  -> GATE 1 FAIL: VR predicts nothing once vol is removed. D IS DEAD.")
    gate_log.append("gate1 orthogonalization: FAIL — VR is vol in disguise / no signal")
    GATE_PASS=False

# --- GATE 2: dispersion vs M7's entropy band --------------------------------
print("\n[GATE 2] DISPERSION — is cross-sectional VR spread wider than M7's 0.046 band?")
if GATE_PASS:
    disp=[]
    for rb,g in vrdf.groupby("rb"):
        v=g["vr"].dropna()
        if len(v)>=MIN_NAMES:
            disp.append(v.quantile(0.9)-v.quantile(0.1))   # p10-p90 spread
    mean_disp=np.mean(disp)
    print(f"  mean cross-sectional VR p10-p90 spread: {mean_disp:.3f}")
    print(f"  (M7 entropy band was {M7_ENTROPY_BAND}; VR is a different scale, but a")
    print(f"   spread of similar tininess vs the per-stock noise std~0.30 = nothing to sort on)")
    # dispersion is meaningful only relative to per-stock noise (std~0.30 at 252 obs)
    g2 = mean_disp > 0.30   # spread must exceed the per-stock estimation noise
    if g2:
        print(f"  -> GATE 2 PASS: VR spread ({mean_disp:.3f}) exceeds per-stock noise (~0.30).")
        gate_log.append("gate2 dispersion: PASS")
    else:
        print(f"  -> GATE 2 FAIL: VR spread ({mean_disp:.3f}) <= per-stock noise (~0.30).")
        print(f"     Cross-sectional VR differences are noise-dominated — nothing to sort on.")
        gate_log.append("gate2 dispersion: FAIL — spread below per-stock estimation noise")
        GATE_PASS=False
else:
    print("  (skipped — gate 1 failed)"); gate_log.append("gate2 dispersion: not reached")

# --- GATE 3: block-bootstrap rank stability ---------------------------------
print("\n[GATE 3] RANK STABILITY — block-bootstrap VR rank across resamples (>0.6)?")
if GATE_PASS:
    def block_bootstrap(x, block=BLOCK_LEN, rng=None):
        x=np.asarray(x,float); n=len(x)
        if rng is None: rng=np.random.default_rng()
        nb=int(np.ceil(n/block)); starts=rng.integers(0,max(1,n-block+1),nb)
        return np.concatenate([x[s:s+block] for s in starts])[:n]
    # multi-month stability, same design as M7 fix
    stab_months=REBAL[12::max(1,len(REBAL[12:])//10)][:10]
    month_stabs=[]
    for stab_rb in stab_months:
        series_map={}
        for tkr,td in tdata.items():
            p=find_pos(td["dates"],stab_rb)
            if p is None or td["flag"][p]!=1 or tkr in CORP_ACTION_SCREEN: continue
            fr=formation_returns(tkr,p)
            if fr is not None: series_map[tkr]=fr
        names=list(series_map.keys())
        if len(names)<MIN_NAMES: continue
        base=np.array([variance_ratio(series_map[t],VR_Q_PRIMARY) for t in names])
        rng=np.random.default_rng(7); corrs=[]
        for _ in range(40):
            boot=np.array([variance_ratio(block_bootstrap(series_map[t],BLOCK_LEN,rng),VR_Q_PRIMARY)
                           for t in names])
            rho=spearmanr(base,boot).correlation
            if not np.isnan(rho): corrs.append(rho)
        if corrs: month_stabs.append(np.mean(corrs))
    rank_stab=np.mean(month_stabs) if month_stabs else np.nan
    print(f"  multi-month block-bootstrap VR rank stability: {rank_stab:.3f} "
          f"(bar {STABILITY_BAR}; M7 failed at {M7_STABILITY})")
    g3 = (not np.isnan(rank_stab)) and rank_stab>STABILITY_BAR
    if g3:
        print(f"  -> GATE 3 PASS: VR ranking stable enough to trade.")
        gate_log.append("gate3 stability: PASS")
    else:
        print(f"  -> GATE 3 FAIL: VR ranking unstable at 252 obs (like M7). D untradeable.")
        gate_log.append(f"gate3 stability: FAIL — rank stability {rank_stab:.3f} < {STABILITY_BAR}")
        GATE_PASS=False
else:
    print("  (skipped — earlier gate failed)"); gate_log.append("gate3 stability: not reached")

print("\n"+"="*78)
print(f"D-GATE RESULT: {'ALL PASS — build variance-ratio version, THEN wavelets if VR shows signal' if GATE_PASS else 'FAILED — D shelved, NO wavelet code'}")
for gl in gate_log: print(f"  {gl}")
print(f"  tally: {SUBTEST_TALLY}")
print("="*78)
save_art(pd.DataFrame({"gate":gate_log}),"d_gate_log")

D-GATE — three sequential checks (surface before any wavelet build)

[coverage] VR panel non-null:
  VR(q=21) coverage: 100.0%

[GATE 1] VOL-ORTHOGONALIZATION — does VR predict returns AFTER removing vol?
  VR coef UNCONTROLLED:              -0.00059 (t=-0.41)
  VR-residual coef +RV+mom+Amihud:   -0.00028 (t=-0.23)
  -> GATE 1 FAIL: VR predicts nothing once vol is removed. D IS DEAD.

[GATE 2] DISPERSION — is cross-sectional VR spread wider than M7's 0.046 band?
  (skipped — gate 1 failed)

[GATE 3] RANK STABILITY — block-bootstrap VR rank across resamples (>0.6)?
  (skipped — earlier gate failed)

D-GATE RESULT: FAILED — D shelved, NO wavelet code
  gate1 orthogonalization: FAIL — VR is vol in disguise / no signal
  gate2 dispersion: not reached
  gate3 stability: not reached
  tally: 32
    -> d_gate_log_2026-08-08.csv


PosixPath('/content/drive/MyDrive/moonshot_compounding/hypotheses/microstructure_anomaly_batch/D_gate_2026-08-08/artifacts/d_gate_log_2026-08-08.csv')

---
## After running — surface before any wavelet build

Report the gate log: gate 1 (orthogonalized VR coefficient + t), gate 2 (VR
dispersion vs the ~0.30 noise floor), gate 3 (block-bootstrap rank stability vs the
0.6 bar / M7's 0.44).

- **Any gate fails** → **D is shelved, no wavelet code** — the pre-agreed accepted
  outcome. Log the failing gate as the finding.
- **All three pass** → build the variance-ratio version first (the economic signal
  in one robust number); only if VR shows genuine signal do we then build MODWT +
  Daubechies LA8 to localize which scale drives it, re-running the same
  orthogonalized FM with boundary-coefficient counts reported.

Writes to `hypotheses/microstructure_anomaly_batch/D_gate_<date>/`.